# 04. Type-Safe Structured Outputs with Pydantic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/google-genai/en/04_structured_outputs.ipynb)

Prompting an LLM to "reply only in JSON" often breaks in production when responses include conversational preambles, trailing commas, or ` ```json ` Markdown code fences. Passing `response_mime_type="application/json"` together with `response_schema=<Pydantic BaseModel>` activates **grammar-constrained decoding**: Gemini restricts token sampling directly to the JSON Schema grammar derived from your model, and `response.parsed` returns an already-validated Pydantic instance with zero brittle regex or string stripping.

![Figure 4-1: Grammar-Constrained Structured Output Pipeline with Pydantic](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_structured_pipeline.png)

*Figure 4-1: Grammar-Constrained Structured Output Pipeline with Pydantic*

---

## 1. Client Setup & Defining a Nested Pydantic Schema

First, set `MODEL = "gemini-3.8-flash"` at the top of the cell so you can switch models in one place, and initialize the `google-genai` client using your `GEMINI_API_KEY` stored in Google Colab Secrets (`🔑` panel on the left sidebar):

In [ ]:
!pip install -q google-genai

In [ ]:
MODEL = "gemini-3.8-flash"

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

Next, compose a nested Pydantic schema (`ActionItem` inside `IncidentSummary`) to extract structured postmortem metadata from a concise incident report. Passing `IncidentSummary` to `response_schema` instructs the SDK to compile the Pydantic model into a strict schema and populate `response.parsed` automatically (`pydantic` comes pre-installed in Google Colab and bundled with `google-genai`):

In [ ]:
from typing import Literal

from google.genai import types
from pydantic import BaseModel, Field


class ActionItem(BaseModel):
    owner: str = Field(description="Team responsible for the follow-up task.")
    task: str = Field(description="Concrete remediation step.")
    priority: Literal["P0", "P1", "P2"] = Field(description="Urgency tier (P0 highest).")


class IncidentSummary(BaseModel):
    title: str = Field(description="Concise incident headline under 60 characters.")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(
        description="Customer-impact severity level."
    )
    root_cause: str = Field(description="One-sentence technical root cause.")
    affected_services: list[str] = Field(description="Impacted service names.")
    action_items: list[ActionItem] = Field(description="Prioritized follow-up tasks.")


response = client.models.generate_content(
    model=MODEL,
    contents="Checkout API returned 502 errors for EU users for 12 minutes due to a Redis timeout.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IncidentSummary,
        temperature=0.1,
    ),
)

# response.parsed is an already-validated IncidentSummary instance with full IDE autocomplete
summary: IncidentSummary = response.parsed
print(f"[{summary.severity}] {summary.title} ({len(summary.action_items)} action items)")
print(summary.model_dump_json(indent=2))

---

## 2. Schema Design Best Practices & Fast Deterministic Classification

While grammar-constrained decoding guarantees syntactic validity, thoughtful Pydantic schema design dramatically improves **semantic accuracy** and **latency** on production workloads:

![Figure 4-2: Pydantic Schema Design Patterns for Reliable LLM Extraction](https://examples.drawlib.com/google-genai/images/04_structured_outputs/ch04_schema_best_practices.png)

*Figure 4-2: Pydantic Schema Design Patterns for Reliable LLM Extraction*

Keep these four engineering rules in mind when authoring schemas for `response_schema`:

1. **Use `Literal` or `Enum` for closed categories**: Constrain finite taxonomies (`Literal["P0", "P1", "P2"]` or `str, Enum`) so downstream routers and database `ENUM` columns never encounter unexpected casing or synonyms.
2. **Compose nested `list[SubModel]` instead of `dict[str, Any]`**: Arbitrary open dictionaries lack fixed property names and cannot be grammar-constrained reliably; model structured records explicitly as `list[SubModel]`.
3. **Attach concise `Field(description=...)` metadata**: Descriptions are forwarded in the compiled JSON Schema, guiding Gemini on domain semantics (such as character budgets, ISO units, or urgency criteria) without bloating your main prompt.
4. **Tune `temperature` and `thinking_level` for deterministic speed**: For straightforward entity extraction and classification on `gemini-3.8-flash`, pairing `temperature=0.1` with `thinking_config=types.ThinkingConfig(thinking_level="low")` minimizes tail latency and output variance.

Putting all four rules together, you can combine a Python `Enum` with `list[RoutedIssue]` and `thinking_level="low"` to split a multi-issue customer ticket into typed routing records and iterate directly over `response.parsed`:

In [ ]:
from enum import Enum


class TicketQueue(str, Enum):
    BILLING = "billing"
    AUTH = "auth"
    ANALYTICS = "analytics"


class RoutedIssue(BaseModel):
    queue: TicketQueue = Field(description="Target support queue.")
    urgent: bool = Field(description="True if payment or login is blocked.")
    summary: str = Field(description="Actionable summary under 50 characters.")


class TriageBatch(BaseModel):
    issues: list[RoutedIssue] = Field(description="Distinct issues found in the message.")


triage_response = client.models.generate_content(
    model=MODEL,
    contents="Customer was billed twice on invoice #8821 and cannot export CSV reports.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TriageBatch,
        temperature=0.1,
        thinking_config=types.ThinkingConfig(thinking_level="low"),
    ),
)

batch: TriageBatch = triage_response.parsed
for item in batch.issues:
    flag = "URGENT" if item.urgent else "NORMAL"
    print(f"[{flag}] -> {item.queue.value}: {item.summary}")